In [1]:
import sys
sys.path.append('../../')
import warnings
import requests
import json
import pyarrow.parquet as pq
from random import randint
from tqdm import trange
import os
import pandas as pd
import re
import csv
import random
import time
from util.get_output import model
now_model=model('../../model')

/home/pai/envs/prototype/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00, 53.90it/s]


In [2]:
def get_output(message):
    messages=[
            {
                "role": "system",
                "content": '''Output format:
Ans: (Option)''',
            },
            {
                "role":"user",
                "content":message,
            }
    ]

    return now_model.get_output(messages)

In [3]:
def get_bias_output(fact):
    template="""
Please answer the question.

Question:%s
A.no   B.yes
"""%(fact)
    return template

In [4]:
jishu1=50
jishu2=50
parquet_file = pq.ParquetFile('./data/test.parquet')
data = parquet_file.read().to_pandas()

In [5]:
i=0
f=open('./data/test.jsonl','a')
while jishu1>0 or jishu2>0:
    if 'no' in data['answer'][i] and jishu1>0:
        now=get_output(get_bias_output(data['question'][i]))
        try:
            now=re.findall(r"Ans([\S|\s]*)",now, re.S|re.I)[0]
        except:
            a=1
        if 'no' in now or 'A' in now or 'No' in now:
            jishu1=jishu1-1
            input_dict={'number':i,'question':data['question'][i],'ans':'A','context':data['context'][i]}
            input_json=json.dumps(input_dict)
            f.write(input_json+'\n')
    elif 'yes' in data['answer'][i] and jishu2>0:
        now=get_output(get_bias_output(data['question'][i]))
        try:
            now=re.findall(r"Ans([\S|\s]*)",now, re.S|re.I)[0]
        except:
            a=1
        if 'yes' in now or 'B' in now or 'Yes' in now:
            jishu2=jishu2-1
            input_dict={'number':i,'question':data['question'][i],'ans':'B','context':data['context'][i]}
            input_json=json.dumps(input_dict)
            f.write(input_json+'\n')
    i=i+1

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end gene

In [6]:
f.close()